# Creating a Simple Neural Network - Iris Flower Classifier  
John Elder, Codemy: https://youtu.be/O9Jx93DAyw4?si=1Pmt5GnwJObynZA3 

### 1. Define Model


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# create model class that inherits nn.Module
class Model(nn.Module):
    # input layer (4 features of iris flower)
    # --> hidden layer H1 (8 neurons)
    # --> H2 (9 neurons)
    # --> output (3 classes of iris flowers)
    
    def __init__(self, in_features=7, h1=8, h2=9, out_features=2):
        super().__init__() # instantiate nn.Module
        self.fc1 = nn.Linear(in_features, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.out = nn.Linear(h2, out_features)

    def forward(self, x):
        x = F.relu(self.fc1(x)) # rectified linear unit activation function
        x = F.relu(self.fc2(x))
        x = self.out(x)
        return x

In [ ]:
# pick manual seed for randomisation
torch.manual_seed(1)

# create instance of model
model = Model()

### 2. Pull Raw Data from GitHub Iris Classifier Dataset


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# url = 'https://gist.githubusercontent.com/netj/8836201/raw/6f9306ad21398ea43cba4f7d537619d0e07d5ae3/iris.csv'
# my_df = pd.read_csv(url)
my_df = pd.read_csv('../data/raw/credit_data.csv')
my_df.tail()

### 3. Preprocess and Normalise Data


In [ ]:
# change labels from string to numeric values (already done. fraud = 1, not fraud = 0)

### 4. Split Preprocessed Dataset into Train (80%) / Test (20%)


In [ ]:
# train test split, set x,y
X = my_df.drop('fraud', axis=1)
y = my_df['fraud']

In [ ]:
# convert to numpy arrays
X = X.values
y = y.values

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2,
    random_state=1,
    stratify=y
)

# y_train

In [ ]:
# convert X features to float tensors
X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)

# convert y labels to long (float) tensors
y_train = torch.LongTensor(y_train.astype(float))
y_test = torch.LongTensor(y_test.astype(float))

### 5. Choose Loss Function and Optimizer


In [ ]:
# set criterion of model to measure error
criterion = nn.CrossEntropyLoss()

# choose Adam optimizer, set learning rate
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

### 6. Train the Model


In [ ]:
# train the model
epochs = 250
losses = []

for i in range(epochs):
    # go forward and get prediction
    y_pred = model.forward(X_train)

    # measure loss/error
    loss = criterion(y_pred, y_train)

    # keep track of losses
    losses.append(loss.item())

    # print every 10 epochs
    if i % 10 == 0:
        print(f'Epoch: {i} Loss: {loss}')

    # backpropagation: take error rate of forward propagation + feed it back through network to fine-tune weights
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [ ]:
# display on a graph
plt.plot(range(epochs), losses)
plt.ylabel('Loss/Error')
plt.xlabel('Epoch')
model.eval()

### 7. Evaluate the Model Using the Test Dataset


In [ ]:
# evaluate model on test dataset
with torch.no_grad():   # turn off back propagation
    y_eval = model.forward(X_test)
    loss = criterion(y_eval, y_test)
loss

In [ ]:
correct = 0
y_pred = []

with torch.no_grad():
    for i, data in enumerate(X_test):
        y_val = model.forward(data)

        print(f'{i+1}.) {str(y_val)} \t predicted: {y_val.argmax().item()} \t actual: {y_test[i]}')

        # correct or not
        if y_val.argmax().item() == y_test[i]:
            correct += 1

        y_pred.append(y_val.argmax().item())
    
print(f'Correct: {correct} / {len(X_test)}')

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# If doing multi-class (0, 1, 2), you must specify average='weighted' or 'macro'
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")

### 8. Applying the Model to New Data


In [ ]:
# new_iris = torch.tensor([6.2, 3.4, 5.4, 2.3])

In [ ]:
# with torch.no_grad():
    # print(model(new_iris))

### 9. How to Save and Load the Model


In [ ]:
# save NN model
# torch.save(model.state_dict(), 'iris_model.pt')

In [ ]:
# load the saved model
# new_model = Model()
# new_model.load_state_dict(torch.load('iris_model.pt'))

In [ ]:
# make sure it loaded correctly
# new_model.eval()